# Chebyshev basis-change truncation experiment

1. it keeps the **BP / RP split** intact:
   - `c000..c054` = BP
   - `c055..c109` = RP
2. it applies the Chebyshev basis change **separately** to BP and RP;
3. for each `K`, it keeps the **first K transformed BP coefficients + first K transformed RP coefficients**, so the feature vector is again of size **2K**;
4. it uses a **numerically stable orthonormal Chebyshev basis** obtained by QR-orthonormalizing the sampled Chebyshev Vandermonde matrix, instead of using a raw full pseudoinverse.

In [9]:

# !pip -q install pyarrow

import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from numpy.polynomial.chebyshev import chebvander

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

CSV_PATH = Path("classification_with_c110_d110_errors_snr.csv")

OUT_DIR = Path.cwd() / "logreg_chebcoef_truncation_out"
RUNS_DIR = OUT_DIR / "runs"
REPORT_DIR = OUT_DIR / "analysis_report"
for d in [OUT_DIR, RUNS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEARTBEAT_PATH = OUT_DIR / "HEARTBEAT.json"
LAST_EVENT_PATH = OUT_DIR / "LAST_EVENT.txt"
MANIFEST_PATH = OUT_DIR / "RUNNING_MANIFEST.json"

def hb(payload):
    payload = dict(payload)
    payload["time"] = time.ctime()
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))

def log_event(msg):
    LAST_EVENT_PATH.write_text(f"[{time.ctime()}] {msg}\n")

# ------------------------
# Experiment config
# ------------------------
K_MIN, K_MAX = 1, 55
K_LIST = list(range(K_MIN, K_MAX + 1))

N_REPEATS = 30
TEST_FRAC = 0.15
VAL_FRAC = 0.15
VAL_FRAC_OF_REMAIN = VAL_FRAC / (1.0 - TEST_FRAC)

BASE_SEED = 1234
THR_GRID_SIZE = 800

LR_PARAMS = dict(
    penalty="l2",
    C=1.0,
    solver="saga",
    class_weight="balanced",
    max_iter=5000,
    tol=1e-4,
    n_jobs=-1,
    random_state=0,
)

manifest = dict(
    started_at=time.ctime(),
    csv=str(CSV_PATH),
    out_dir=str(OUT_DIR.resolve()),
    plan=dict(
        K_MIN=K_MIN, K_MAX=K_MAX, N_REPEATS=N_REPEATS,
        TEST_FRAC=TEST_FRAC, VAL_FRAC=VAL_FRAC,
        THR_GRID_SIZE=THR_GRID_SIZE,
        LR_PARAMS=LR_PARAMS,
        basis_change="orthonormal Chebyshev basis in coefficient space, separately for BP and RP",
    ),
)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
log_event("Manifest created.")
hb({"status": "ready"})

print("OUT_DIR:", OUT_DIR.resolve())


OUT_DIR: /Users/kris/Desktop/Astrophysics/logreg_chebcoef_truncation_out


In [10]:

df = pd.read_csv(CSV_PATH)
if "y" not in df.columns:
    raise KeyError("CSV must contain 'y'")

df["y"] = df["y"].astype(int)

C110 = [f"c{i:03d}" for i in range(110)]
missing = [c for c in C110 if c not in df.columns]
if missing:
    raise KeyError(f"Missing c110 columns, e.g. {missing[:6]}")

# c000..c054 = BP, c055..c109 = RP
BP = [f"c{i:03d}" for i in range(55)]
RP = [f"c{i:03d}" for i in range(55, 110)]

Xbp = df[BP].to_numpy(np.float64)
Xrp = df[RP].to_numpy(np.float64)
y_all = df["y"].to_numpy(int)

print("Xbp:", Xbp.shape, "Xrp:", Xrp.shape, "pos_rate:", float(y_all.mean()))


Xbp: (2815, 55) Xrp: (2815, 55) pos_rate: 0.19822380106571935


In [11]:

# --------------------------------------------------
# Coefficient-space Chebyshev basis change
# --------------------------------------------------
# We define a discrete Chebyshev basis over coefficient index, separately for BP and RP.
# To avoid the numerical instability of a raw Vandermonde pseudoinverse, we QR-orthonormalize
# the sampled Chebyshev Vandermonde matrix. This yields an orthonormal basis Q whose columns
# span the same 55-dimensional subspace.
#
# For a row vector x (1 x 55), the coordinates in the new basis are:
#   z = x @ Q
# and the exact reconstruction is:
#   x = z @ Q.T

def build_orthonormal_cheb_basis(n=55):
    x = np.linspace(-1.0, 1.0, n)
    V = chebvander(x, deg=n-1)          # shape (n, n)
    Q, R = np.linalg.qr(V)              # orthonormal columns
    return x, V, Q, R

x_bp, V_bp, Q_bp, R_bp = build_orthonormal_cheb_basis(55)
x_rp, V_rp, Q_rp, R_rp = build_orthonormal_cheb_basis(55)

# Transform the coefficient blocks separately
Zbp = Xbp @ Q_bp     # coordinates in BP Chebyshev basis
Zrp = Xrp @ Q_rp     # coordinates in RP Chebyshev basis

bp_orth_err = float(np.linalg.norm(Q_bp.T @ Q_bp - np.eye(55)))
rp_orth_err = float(np.linalg.norm(Q_rp.T @ Q_rp - np.eye(55)))

bp_recon = Zbp @ Q_bp.T
rp_recon = Zrp @ Q_rp.T

bp_max_abs_recon_err = float(np.max(np.abs(bp_recon - Xbp)))
rp_max_abs_recon_err = float(np.max(np.abs(rp_recon - Xrp)))

basis_audit = pd.DataFrame([{
    "bp_orth_err_fro": bp_orth_err,
    "rp_orth_err_fro": rp_orth_err,
    "bp_max_abs_reconstruction_error": bp_max_abs_recon_err,
    "rp_max_abs_reconstruction_error": rp_max_abs_recon_err,
}])

display(basis_audit)

basis_audit.to_csv(OUT_DIR / "cheb_basis_audit.csv", index=False)
np.save(OUT_DIR / "Q_bp_cheb_orthonormal.npy", Q_bp)
np.save(OUT_DIR / "Q_rp_cheb_orthonormal.npy", Q_rp)

print("Saved basis audit + transform matrices.")


,bp_orth_err_fro,rp_orth_err_fro,bp_max_abs_reconstruction_error,rp_max_abs_reconstruction_error
0,4.914827e-15,4.914827e-15,8.881784e-16,7.771561e-16


Saved basis audit + transform matrices.


In [12]:

def make_splits(y, n_repeats, base_seed):
    splitter = StratifiedShuffleSplit(n_splits=n_repeats, test_size=TEST_FRAC, random_state=base_seed)
    idx_all = np.arange(len(y))
    splits = []
    for r, (trainval_rel, test_rel) in enumerate(splitter.split(np.zeros(len(y)), y)):
        idx_trainval = idx_all[trainval_rel]
        idx_test = idx_all[test_rel]

        y_tv = y[idx_trainval]
        splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC_OF_REMAIN, random_state=base_seed + 1000 + r)
        tr_rel2, va_rel2 = next(splitter2.split(np.zeros(len(y_tv)), y_tv))

        idx_train = idx_trainval[tr_rel2]
        idx_val   = idx_trainval[va_rel2]
        splits.append(dict(repeat=r, train_idx=idx_train, val_idx=idx_val, test_idx=idx_test))
    return splits

splits = make_splits(y_all, N_REPEATS, BASE_SEED)
print("splits:", len(splits), "sizes:", {k: len(splits[0][k]) for k in ["train_idx","val_idx","test_idx"]})


splits: 30 sizes: {'train_idx': 1969, 'val_idx': 423, 'test_idx': 423}


In [13]:

def make_X_for_K(idx, K):
    bp = Zbp[idx, :K]   # (N, K)
    rp = Zrp[idx, :K]   # (N, K)
    X = np.concatenate([bp, rp], axis=1).astype(np.float32, copy=False)  # (N, 2K)
    return X

Xdemo = make_X_for_K(np.arange(5), 10)
print("demo shape:", Xdemo.shape)


demo shape: (5, 20)


In [14]:

def prob_metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    pc = np.clip(p, 1e-15, 1-1e-15)
    return dict(
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        PR_AUC=average_precision_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        Brier=brier_score_loss(y_true, p),
        LogLoss=log_loss(y_true, pc, labels=[0,1]),
    )

def confusion_metrics(y_true, y_hat):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return prec, sens, spec, tn, fp, fn, tp

def threshold_grid(p, n=800):
    lo, hi = float(np.min(p)), float(np.max(p))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return np.array([0.5])
    return np.linspace(lo, hi, n)

def choose_threshold(y_true, p, criterion="youden", n=800):
    best_thr, best_score = 0.5, -np.inf
    for thr in threshold_grid(p, n):
        yhat = (p >= thr).astype(int)
        prec, sens, spec, *_ = confusion_metrics(y_true, yhat)
        if criterion == "youden":
            score = (sens + spec - 1) if np.isfinite(sens) and np.isfinite(spec) else -np.inf
        elif criterion == "f1":
            score = (2*prec*sens/(prec+sens)) if np.isfinite(prec) and np.isfinite(sens) and (prec+sens)>0 else -np.inf
        else:
            raise ValueError
        if score > best_score:
            best_score, best_thr = float(score), float(thr)
    return best_thr, float(best_score)

def evaluate(yv, pv, yt, pt):
    out = {f"test_{k}": float(v) for k,v in prob_metrics(yt, pt).items()}
    for crit in ["youden", "f1"]:
        thr, sc = choose_threshold(yv, pv, crit, THR_GRID_SIZE)
        yhat = (pt >= thr).astype(int)
        prec, sens, spec, tn, fp, fn, tp = confusion_metrics(yt, yhat)
        out[f"{crit}_val_thr"] = float(thr)
        out[f"{crit}_val_score"] = float(sc)
        out[f"{crit}_test_Precision"] = float(prec)
        out[f"{crit}_test_Sensitivity"] = float(sens)
        out[f"{crit}_test_Specificity"] = float(spec)
        out[f"{crit}_test_tn"] = int(tn); out[f"{crit}_test_fp"] = int(fp)
        out[f"{crit}_test_fn"] = int(fn); out[f"{crit}_test_tp"] = int(tp)
    return out


In [15]:

def run_id(K, rep):
    return f"K{int(K):02d}__r{int(rep):02d}"

def shard_path(K, rep):
    return RUNS_DIR / f"{run_id(K, rep)}.parquet"

def done(K, rep):
    return shard_path(K, rep).exists()

def save_row(row):
    pd.DataFrame([row]).to_parquet(RUNS_DIR / f"{row['run_id']}.parquet", index=False)

def load_all_rows():
    files = sorted(RUNS_DIR.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

print("Existing shards:", len(list(RUNS_DIR.glob("*.parquet"))))


Existing shards: 125


In [ ]:

pK = tqdm(K_LIST, desc="K (Chebyshev coeffs per BP/RP)", leave=True)

for K in pK:
    pK.set_postfix_str(f"K={K}")
    log_event(f"Starting K={K}")

    p_rep = tqdm(splits, desc=f"Repeats [K={K}]", leave=False)
    for sp in p_rep:
        rep = sp["repeat"]
        if done(K, rep):
            continue

        hb({"status": "run_start", "K": int(K), "repeat": int(rep)})

        tr, va, te = sp["train_idx"], sp["val_idx"], sp["test_idx"]

        Xtr = make_X_for_K(tr, K)
        Xva = make_X_for_K(va, K)
        Xte = make_X_for_K(te, K)

        ytr, yva, yte = y_all[tr], y_all[va], y_all[te]

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xva_s = scaler.transform(Xva)
        Xte_s = scaler.transform(Xte)

        t0 = time.time()
        clf = LogisticRegression(**LR_PARAMS)
        clf.fit(Xtr_s, ytr)
        pva = clf.predict_proba(Xva_s)[:, 1]
        pte = clf.predict_proba(Xte_s)[:, 1]
        elapsed = time.time() - t0

        row = {
            "run_id": run_id(K, rep),
            "K": int(K),
            "repeat": int(rep),
            "runtime_s": float(elapsed),
            "n_train": int(len(tr)),
            "n_val": int(len(va)),
            "n_test": int(len(te)),
            "feature_dim": int(Xtr_s.shape[1]),
        }
        row.update(evaluate(yva, pva, yte, pte))
        save_row(row)

        hb({"status": "run_done", "K": int(K), "repeat": int(rep), "runtime_s": float(elapsed)})

log_event("All runs complete.")
hb({"status": "complete"})


In [17]:

res = load_all_rows()
print("rows:", res.shape)
display(res.head(5))

res.to_parquet(OUT_DIR / "truncation_logreg_chebcoef_raw.parquet", index=False)
res.to_csv(OUT_DIR / "truncation_logreg_chebcoef_raw.csv", index=False)
print("Saved raw outputs.")

metrics = [
    "runtime_s",
    "test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
    "youden_test_Precision","youden_test_Sensitivity","youden_test_Specificity","youden_val_thr",
    "f1_test_Precision","f1_test_Sensitivity","f1_test_Specificity","f1_val_thr"
]

summ = res.groupby("K", as_index=False)[metrics].agg(["mean","std"])
summ.columns = [c if s == "" else f"{c}_{s}" for c, s in summ.columns]
summ.to_csv(OUT_DIR / "truncation_logreg_chebcoef_summary_byK.csv", index=False)
print("Saved summary CSV.")
display(summ.head(10))


rows: (1720, 30)


,run_id,K,repeat,runtime_s,n_train,n_val,n_test,feature_dim,test_ROC_AUC,test_PR_AUC,...,youden_test_tp,f1_val_thr,f1_val_score,f1_test_Precision,f1_test_Sensitivity,f1_test_Specificity,f1_test_tn,f1_test_fp,f1_test_fn,f1_test_tp
0,K01__r00,1,0,0.002926,1969,423,423,2,0.925622,0.836815,...,70,0.525609,0.787500,0.758621,0.785714,0.938053,318,21,18,66
1,K01__r01,1,1,0.001988,1969,423,423,2,0.894508,0.817502,...,68,0.648780,0.792208,0.804878,0.785714,0.952802,323,16,18,66
2,K01__r02,1,2,0.001800,1969,423,423,2,0.828136,0.730389,...,58,0.607036,0.829545,0.756757,0.666667,0.946903,321,18,28,56
3,K01__r03,1,3,0.001609,1969,423,423,2,0.931205,0.850737,...,72,0.650328,0.812500,0.807229,0.797619,0.952802,323,16,17,67
4,K01__r04,1,4,0.001781,1969,423,423,2,0.903111,0.856601,...,73,0.621655,0.817610,0.862500,0.821429,0.967552,328,11,15,69


Saved raw outputs.
Saved summary CSV.


,K,runtime_s_mean,runtime_s_std,test_PR_AUC_mean,test_PR_AUC_std,test_ROC_AUC_mean,test_ROC_AUC_std,test_LogLoss_mean,test_LogLoss_std,test_Brier_mean,...,youden_val_thr_mean,youden_val_thr_std,f1_test_Precision_mean,f1_test_Precision_std,f1_test_Sensitivity_mean,f1_test_Sensitivity_std,f1_test_Specificity_mean,f1_test_Specificity_std,f1_val_thr_mean,f1_val_thr_std
0,1,0.001944,0.000924,0.835176,0.035651,0.903949,0.025912,0.329166,0.023972,0.089100,...,0.481724,0.084800,0.811825,0.053926,0.755238,0.057164,0.955310,0.017028,0.621104,0.081335
1,2,0.009676,0.000734,0.850813,0.035952,0.908126,0.027193,0.308525,0.023277,0.080682,...,0.464759,0.068035,0.837837,0.052250,0.764286,0.065757,0.962537,0.014350,0.617293,0.091562
2,3,0.012310,0.000932,0.853663,0.035882,0.910794,0.027220,0.303677,0.023757,0.078555,...,0.471519,0.100129,0.851601,0.046653,0.766270,0.059193,0.966470,0.012285,0.621443,0.088934
3,4,0.019993,0.001087,0.852557,0.036728,0.910302,0.027792,0.302791,0.024410,0.077973,...,0.444618,0.088963,0.856628,0.045467,0.769841,0.059756,0.967748,0.011036,0.622782,0.073738
4,5,0.027635,0.002453,0.852492,0.036621,0.909924,0.027658,0.302362,0.024450,0.077713,...,0.447159,0.094814,0.856135,0.048643,0.768254,0.063898,0.967453,0.013180,0.622297,0.077641
5,6,0.034912,0.005049,0.851364,0.037603,0.909938,0.027741,0.302963,0.024998,0.077612,...,0.431737,0.090043,0.854779,0.048614,0.769048,0.058955,0.967060,0.013509,0.633296,0.079107
6,7,0.044289,0.006666,0.851980,0.037174,0.909602,0.027481,0.302618,0.025041,0.077466,...,0.431494,0.085425,0.851445,0.048290,0.769841,0.066491,0.966077,0.013765,0.620922,0.084160
7,8,0.052317,0.002695,0.853809,0.037020,0.912248,0.026766,0.300614,0.025934,0.077017,...,0.429691,0.086066,0.850075,0.048905,0.768254,0.067469,0.965683,0.013089,0.628771,0.082399
8,9,0.072759,0.006261,0.853176,0.037216,0.912071,0.026608,0.301213,0.026012,0.077221,...,0.434833,0.086694,0.852708,0.051085,0.763095,0.069511,0.966667,0.013286,0.635470,0.090103
9,10,0.099153,0.010586,0.855326,0.038382,0.911357,0.028505,0.298614,0.026530,0.076573,...,0.438533,0.087884,0.851726,0.044236,0.770635,0.073714,0.966077,0.011869,0.622493,0.091590


In [19]:

import plotly.graph_objects as go

def line_with_band(metric):
    dfm = res.groupby("K")[metric].agg(["mean","std"]).reset_index().sort_values("K")
    dfm.columns = ["K","mean","std"]
    dfm["lo"] = dfm["mean"] - dfm["std"]
    dfm["hi"] = dfm["mean"] + dfm["std"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["mean"], mode="lines+markers", name=f"{metric} mean"))
    fig.add_trace(go.Scatter(
        x=pd.concat([dfm["K"], dfm["K"][::-1]]),
        y=pd.concat([dfm["hi"], dfm["lo"][::-1]]),
        fill="toself", line=dict(width=0), opacity=0.22, name="±1 std"
    ))
    fig.update_layout(
        title=f"{metric} vs K (mean ± std across repeats)",
        xaxis_title="K (Chebyshev coeffs per BP/RP)",
        yaxis_title=metric,
        template="plotly_white",
        width=2000, height=900,
        showlegend=True,
    )
    return fig

line_with_band("test_PR_AUC").show()
line_with_band("test_LogLoss").show()
